In [11]:
import pandas as pd
import numpy as np

In [12]:
billing_df=pd.read_csv("/content/billing.csv")
cities_df=pd.read_csv("/content/cities.csv")
complaints_df=pd.read_csv("/content/complaints.csv")
contracts_df=pd.read_csv("/content/contracts.csv")
customer_feedback_df=pd.read_csv("/content/customer_feedback.csv")

customers_df=pd.read_csv("/content/customers.csv")
data_quality_df=pd.read_csv("/content/data_quality_issue_log.csv")
devices_df=pd.read_csv("/content/devices.csv")
employees_df=pd.read_csv("/content/employees.csv")
marketing_campaigns_df=pd.read_csv("/content/marketing_campaigns.csv")

network_quality_df=pd.read_csv("/content/network_quality.csv")
payments_df=pd.read_csv("/content/payments.csv")
plan_historydf=pd.read_csv("/content/plan_history.csv")
plans_df=pd.read_csv("/content/plans.csv")
recharges_df=pd.read_csv("/content/recharges.csv")

regions_df=pd.read_csv("/content/regions.csv")
retention_campaign_df=pd.read_csv("/content/retention_campaigns.csv")
states_df=pd.read_csv("/content/states.csv")
stores_df=pd.read_csv("/content/stores.csv")
subscriptions_df=pd.read_csv("/content/subscriptions.csv")

support_tickets_df=pd.read_csv("/content/support_tickets.csv")
usage_data_df=pd.read_csv("/content/usage_data.csv")
usage_sms_df=pd.read_csv("/content/usage_sms.csv")
usage_voice_df=pd.read_csv("/content/usage_voice.csv")

In [13]:
cust = customers_df.copy()
subs = subscriptions_df.copy()
ticket = support_tickets_df.copy()
contract = contracts_df.copy()

In [14]:
date_columns = {
    "customers": (cust, ["acquisition_date", "churn_date"]),
    "subscriptions": (subs, ["start_date", "end_date"]),
    "support_tickets": (ticket, ["created_date", "resolution_date"]),
    "contracts": (contract, ["start_date", "end_date"])
}
for df, columns in date_columns.values():
    for col in columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

In [15]:
# Define current active customers using customer_status
cust["status_clean"] = (
    cust["customer_status"].astype(str).str.strip().str.lower()
)

active_statuses = ["active"]
cust["is_active"] = cust["status_clean"].isin(active_statuses)
# Define churned customers using customer_status
cust["is_churned"] = cust["status_clean"].isin(["churned", "churn"])


In [16]:
# Helper function for safe percentage calculation
def safe_percent(numerator, denominator):
    if denominator == 0:
        return np.nan
    return numerator / denominator * 100

**1. Customer Churn Rate**

In [19]:
# Customer Churn Rate

total_customers = customers_df["customer_id"].nunique()

churned_customers = customers_df[
    customers_df["customer_status"].str.lower().str.strip()
    .isin(["churned", "churn"])
]["customer_id"].nunique()

churn_rate = (
    churned_customers / total_customers * 100
    if total_customers > 0 else 0
)

print("Total Customers:", total_customers)
print("Churned Customers:", churned_customers)
print("Customer Churn Rate:", round(churn_rate, 2), "%")

Total Customers: 19000
Churned Customers: 3506
Customer Churn Rate: 18.45 %


**2. Customer Retention Rate**

In [20]:
# Customer Retention Rate

start_date = "2026-09-01"
end_date = "2026-10-01"

start_date = pd.to_datetime(start_date)
end_date = pd.to_datetime(end_date)

customers_df["acquisition_date"] = pd.to_datetime(
    customers_df["acquisition_date"], errors="coerce", dayfirst=True
)
customers_df["churn_date"] = pd.to_datetime(
    customers_df["churn_date"], errors="coerce", dayfirst=True
)

start_customers = customers_df[
    (customers_df["acquisition_date"] < start_date) &
    (
        customers_df["churn_date"].isna() |
        (customers_df["churn_date"] >= start_date)
    )
]["customer_id"].nunique()

new_customers = customers_df[
    (customers_df["acquisition_date"] >= start_date) &
    (customers_df["acquisition_date"] < end_date)
]["customer_id"].nunique()

end_customers = customers_df[
    (customers_df["acquisition_date"] < end_date) &
    (
        customers_df["churn_date"].isna() |
        (customers_df["churn_date"] >= end_date)
    )
]["customer_id"].nunique()

retention_rate = (
    (end_customers - new_customers) / start_customers * 100
    if start_customers > 0 else 0
)

print("Starting Customers:", start_customers)
print("New Customers:", new_customers)
print("Ending Customers:", end_customers)
print("Customer Retention Rate:", round(retention_rate, 2), "%")

Starting Customers: 15185
New Customers: 0
Ending Customers: 15185
Customer Retention Rate: 100.0 %


**3. ARPU — Average Revenue Per User**

In [21]:
# ARPU

active_subs = subscriptions_df[
    subscriptions_df["subscription_status"].str.lower().str.strip()
    == "active"
].copy()

active_subs["monthly_charge_inr"] = pd.to_numeric(
    active_subs["monthly_charge_inr"], errors="coerce"
)

mrr = active_subs["monthly_charge_inr"].sum()

active_customers = customers_df[
    customers_df["customer_status"].str.lower().str.strip() == "active"
]["customer_id"].nunique()

arpu = mrr / active_customers if active_customers > 0 else 0

print("Monthly Recurring Revenue: ₹", round(mrr, 2))
print("Active Customers:", active_customers)
print("ARPU: ₹", round(arpu, 2))

Monthly Recurring Revenue: ₹ 14920323.0
Active Customers: 15494
ARPU: ₹ 962.97


**4. Customer Lifetime Value (CLV)**

In [22]:
# Customer Lifetime Value

gross_margin_pct = 40  # Example assumption: 40%

monthly_churn_pct = 2.1  # Example assumption: 2.1%

gross_margin = gross_margin_pct / 100
monthly_churn_rate = monthly_churn_pct / 100

clv = (
    arpu * gross_margin / monthly_churn_rate
    if monthly_churn_rate > 0 else 0
)

print("ARPU: ₹", round(arpu, 2))
print("Gross Margin:", gross_margin_pct, "%")
print("Monthly Churn Rate:", monthly_churn_pct, "%")
print("Estimated Customer Lifetime Value: ₹", round(clv, 2))

ARPU: ₹ 962.97
Gross Margin: 40 %
Monthly Churn Rate: 2.1 %
Estimated Customer Lifetime Value: ₹ 18342.37


**5. Revenue Lost to Churn**

In [23]:
# Revenue Lost to Churn

churned = customers_df[
    customers_df["customer_status"].str.lower().str.strip()
    .isin(["churned", "churn"])
].copy()

plan_charges = plans_df[["plan_id", "monthly_charge"]].copy()

plan_charges["monthly_charge"] = pd.to_numeric(
    plan_charges["monthly_charge"], errors="coerce"
)

churned_revenue = churned.merge(
    plan_charges,
    on="plan_id",
    how="left"
)

revenue_lost = churned_revenue.groupby(
    "customer_id"
)["monthly_charge"].first().sum()

print("Churned Customers:", churned["customer_id"].nunique())
print("Estimated Monthly Revenue Lost: ₹", round(revenue_lost, 2))

Churned Customers: 3506
Estimated Monthly Revenue Lost: ₹ 2254618


**6. Monthly Recurring Revenue (MRR)**

In [24]:
# Monthly Recurring Revenue

active_subs = subscriptions_df[
    subscriptions_df["subscription_status"].str.lower().str.strip()
    == "active"
].copy()

active_subs["monthly_charge_inr"] = pd.to_numeric(
    active_subs["monthly_charge_inr"], errors="coerce"
)

mrr = active_subs["monthly_charge_inr"].sum()

print("Active Subscriptions:", len(active_subs))
print("Monthly Recurring Revenue: ₹", round(mrr, 2))

Active Subscriptions: 16930
Monthly Recurring Revenue: ₹ 14920323.0


**7. First Contact Resolution (FCR)**

In [25]:
# First Contact Resolution

fcr_values = (
    support_tickets_df["first_contact_resolution"]
    .astype(str).str.lower().str.strip()
)

resolved_first = fcr_values.isin(["true", "1", "yes", "resolved"])
not_resolved_first = fcr_values.isin(["false", "0", "no"])

valid_tickets = resolved_first | not_resolved_first

fcr_rate = (
    resolved_first.sum() / valid_tickets.sum() * 100
    if valid_tickets.sum() > 0 else 0
)

print("Total Valid Tickets:", valid_tickets.sum())
print("Resolved on First Contact:", resolved_first.sum())
print("First Contact Resolution Rate:", round(fcr_rate, 2), "%")

Total Valid Tickets: 31490
Resolved on First Contact: 12726
First Contact Resolution Rate: 40.41 %


In [26]:
print(support_tickets_df["first_contact_resolution"].value_counts(dropna=False))

first_contact_resolution
No     18764
Yes    12726
Name: count, dtype: int64


**8. Average Resolution Time**

In [28]:
# Average Resolution Time

resolved_tickets = support_tickets_df[
    support_tickets_df["resolution_date"].notna()
].copy()

resolved_tickets["resolution_hours"] = pd.to_numeric(
    resolved_tickets["resolution_hours"], errors="coerce"
)

avg_resolution_time = resolved_tickets[
    "resolution_hours"
].mean()

print("Resolved Tickets:", len(resolved_tickets))
print("Average Resolution Time:", round(avg_resolution_time, 2), "hours")

Resolved Tickets: 18124
Average Resolution Time: 16.04 hours


**9. Contract Renewal Rate**

In [29]:
# Contract Renewal Rate

renewal_status = (
    contracts_df["renewal_status"].astype(str).str.lower().str.strip()
)

renewed = renewal_status.isin(["renewed", "yes", "true", "1"])

due_for_renewal = renewal_status.isin([
    "renewed", "not renewed", "pending", "due",
    "yes", "no", "true", "false", "1", "0"
])

renewal_rate = (
    (renewed & due_for_renewal).sum() / due_for_renewal.sum() * 100
    if due_for_renewal.sum() > 0 else 0
)

print("Contracts Due for Renewal:", due_for_renewal.sum())
print("Contracts Renewed:", (renewed & due_for_renewal).sum())
print("Contract Renewal Rate:", round(renewal_rate, 2), "%")

Contracts Due for Renewal: 6094
Contracts Renewed: 3676
Contract Renewal Rate: 60.32 %


In [ ]:
print(contracts["renewal_status"].value_counts(dropna=False))

**10. Average Customer Tenure**

In [30]:
# Average Customer Tenure

active_customers = customers_df[
    customers_df["customer_status"].str.lower().str.strip() == "active"
].copy()

active_customers["tenure_months"] = pd.to_numeric(
    active_customers["tenure_months"], errors="coerce"
)

avg_tenure = active_customers["tenure_months"].mean()

print("Active Customers:", active_customers["customer_id"].nunique())
print("Average Customer Tenure:", round(avg_tenure, 2), "months")

Active Customers: 15494
Average Customer Tenure: 87.1 months


In [31]:
# Create a copy
cohort_df = customers_df.copy()

# Convert dates to datetime
cohort_df["acquisition_date"] = pd.to_datetime(
    cohort_df["acquisition_date"],
    errors="coerce",
    dayfirst=True
)

cohort_df["churn_date"] = pd.to_datetime(
    cohort_df["churn_date"],
    errors="coerce",
    dayfirst=True
)

# Remove rows without an acquisition date or customer ID
cohort_df = cohort_df.dropna(
    subset=["customer_id", "acquisition_date"]
)

# Keep one record per customer
cohort_df = cohort_df.drop_duplicates(
    subset=["customer_id"]
)

# Create acquisition month (cohort)
cohort_df["cohort_month"] = (
    cohort_df["acquisition_date"].dt.to_period("M")
)

print(cohort_df[
    ["customer_id", "acquisition_date", "churn_date", "cohort_month"]
].head())

  customer_id acquisition_date churn_date cohort_month
0    C0000001       2026-01-15 2026-03-22      2026-01
1    C0000002       2014-01-01        NaT      2014-01
2    C0000003       2015-11-11 2015-12-22      2015-11
4    C0000005       2014-10-20 2023-10-22      2014-10
5    C0000006       2018-06-20        NaT      2018-06


In [32]:
# Calculate each customer's age in months at each milestone

for months in [3, 6, 12]:
    milestone_date = cohort_df["acquisition_date"] + pd.DateOffset(
        months=months
    )

    cohort_df[f"active_{months}m"] = (
        cohort_df["churn_date"].isna() |
        (cohort_df["churn_date"] >= milestone_date)
    )

    # Exclude customers who have not yet reached the milestone
    cohort_df.loc[
        milestone_date > pd.Timestamp.today(),
        f"active_{months}m"
    ] = np.nan

print(cohort_df[
    ["customer_id", "cohort_month",
     "active_3m", "active_6m", "active_12m"]
].head())

  customer_id cohort_month active_3m active_6m active_12m
0    C0000001      2026-01     False     False        NaN
1    C0000002      2014-01      True      True       True
2    C0000003      2015-11     False     False      False
4    C0000005      2014-10      True      True       True
5    C0000006      2018-06      True      True       True


/tmp/ipykernel_4742/4227196809.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  cohort_df.loc[
/tmp/ipykernel_4742/4227196809.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  cohort_df.loc[
/tmp/ipykernel_4742/4227196809.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  cohort_df.loc[


In [33]:
# Calculate the share retained within each acquisition cohort

retention_table = cohort_df.pivot_table(
    index="cohort_month",
    values=["active_3m", "active_6m", "active_12m"],
    aggfunc="mean"
)

# Convert shares into percentages
retention_table = retention_table * 100

# Rename columns
retention_table.columns = [
    "Retention after 3 months (%)",
    "Retention after 6 months (%)",
    "Retention after 12 months (%)"
]

print(retention_table.round(2))

             Retention after 3 months (%) Retention after 6 months (%)  \
cohort_month                                                             
2014-01                         99.311663                    99.847036   
2014-02                             100.0                        100.0   
2014-03                             100.0                        100.0   
2014-04                             100.0                        100.0   
2014-05                         98.924731                    98.924731   
...                                   ...                          ...   
2025-11                               NaN                    77.876106   
2025-12                               NaN                    74.285714   
2026-01                               NaN                    56.363636   
2026-02                               NaN                    46.666667   
2026-03                               NaN                    33.333333   

             Retention after 12 month